# Поиск изображения: матрица сходства

Дополнение к § 3.5. Сначала разберём матрицу на учебных векторах, затем запустим готовый поиск на открытой галерее. В конце можно заменить изображения собственными фотографиями.

## Два режима работы

Первые ячейки с учебными векторами работают без загрузки весов. Для запуска CLIP установите зависимости из `requirements.txt` и распакуйте `images.zip` рядом с ноутбуком: архив создаёт папку `data/`. В Colab выполните `%pip install -r requirements.txt` в отдельной ячейке. Затем измените `run_model = False` на `True` и выполните ячейки сверху вниз. Первый запуск скачает веса и будет работать на CPU. Не распаковывайте архив внутрь уже созданной папки `data/`.

## Матрица оценок

Строки соответствуют изображениям, столбцы — описаниям. Отсортируйте оценки каждого столбца, чтобы найти лучший снимок для соответствующего текста.

In [ ]:
import numpy as np
images = np.array([[0.8,0.6],[0,1],[1,0]], dtype=float)
texts = np.array([[1,0],[0,1]], dtype=float)
images /= np.linalg.norm(images,axis=1,keepdims=True)
texts /= np.linalg.norm(texts,axis=1,keepdims=True)
scores = images @ texts.T
print(scores.round(3), scores.shape)
assert scores.shape == (3,2)

## Новый текст — новый порядок

Здесь меняется только вектор текста, а список изображений остаётся прежним. Сравните ранги и объясните, что изменилось из-за нового запроса.

In [ ]:
new_text = np.array([0.6,0.8])
new_text /= np.linalg.norm(new_text)
print(np.argsort(-(images @ new_text)).tolist())

## Нет подходящего кандидата

Даже для постороннего описания программа вернёт номер изображения с наибольшей оценкой. Почему одного ранжирования недостаточно, чтобы сообщить, что совпадений нет?

In [ ]:
unrelated = np.array([-1.0,-1.0]); unrelated /= np.linalg.norm(unrelated)
print("Лидер среди кандидатов:", int(np.argmax(images @ unrelated)))

## Готовая галерея и собственные фотографии

При `run_model = True` код ниже ищет по шести изображениям из архива: чашке и её обрезке, кошке, ракете, кирпичной стене и траве. Запросы про чашку, кошку и ракету имеют заранее отмеченные совпадения, а запрос про велосипед — нет. После этого замените `paths` своими шестью фотографиями, запросы и ручную разметку обновите согласованно. Используйте изображения без персональных данных.

In [ ]:
run_model = False
if run_model:
    from PIL import Image
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    from transformers import CLIPModel, CLIPProcessor

    paths = ["data/coffee.png", "data/coffee-crop.png", "data/chelsea.png",
             "data/rocket.jpg", "data/brick.png", "data/grass.png"]
    queries = ["a photo of a cup of coffee", "a photo of a cat",
               "a photo of a rocket launch", "a photo of a bicycle"]
    # До запуска отмечаем подходящие файлы; у велосипеда совпадений нет.
    relevant = [{paths[0], paths[1]}, {paths[2]}, {paths[3]}, set()]
    model_id = "openai/clip-vit-base-patch32"
    revision = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
    processor = CLIPProcessor.from_pretrained(model_id, revision=revision)
    model = CLIPModel.from_pretrained(model_id, revision=revision).eval()
    images = [Image.open(path).convert("RGB") for path in paths]
    def search(texts):
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
        with torch.inference_mode():
            matrix = model(**inputs).logits_per_image
        return pd.DataFrame(matrix.numpy(), index=paths, columns=texts)
    table = search(queries)
    assert table.shape == (6, 4)
    print(table.round(2).to_string())
    fig, axes = plt.subplots(len(queries), 3, figsize=(9, 10))
    hits = []
    for row, (query, allowed) in enumerate(zip(queries, relevant)):
        top = table[query].sort_values(ascending=False).head(3).index.tolist()
        if allowed: hits.append(bool(set(top) & allowed))
        print(query, top, "попадание:" if allowed else "нет подходящего фото:",
              bool(set(top) & allowed) if allowed else "ранжирование не умеет отказываться")
        for rank, (ax, path) in enumerate(zip(axes[row], top), start=1):
            ax.imshow(images[paths.index(path)])
            ax.set_title(f"{row+1}.{rank}", fontsize=18)
            ax.axis("off")
    plt.tight_layout(); plt.show()
    print("Попадание в top-3 (только запросы с совпадениями):", sum(hits)/len(hits))
    changed = search(["a close-up photo of a white coffee cup on a wooden table"])
    print("Новая формулировка:", changed.iloc[:, 0].sort_values(ascending=False).index.tolist())


## Проверяем порядок

Для каждого запроса выведите top-3 и сравните с ручной разметкой. Затем измените только формулировку одного запроса и пересчитайте матрицу. Запишите изменение мест, а не только оценку лидера. Если подходящего фото нет, отметьте, что CLIP всё равно поставил что-то первым.


In [ ]:
if run_model:
    for query in queries:
        print(query, table[query].sort_values(ascending=False).head(3).index.tolist())


In [ ]:
if run_model:
    from importlib.metadata import version
    print("Библиотеки:", {name: version(name) for name in
          ["torch", "torchvision", "transformers", "numpy", "pandas"]})
    print("Модель и ревизия:", model_id, revision)


## Самопроверка




### 1. Как CLIP оценивает сочетание фотографии и описания?

A. Сравнивает число пикселей с числом слов.

B. Сопоставляет векторы изображения и текста.

C. Выбирает класс только из заранее заданных меток.

D. Сравнивает размеры файлов.




### 2. Вектор запроса равен $(1,0)$, а вектор снимка — $(0{,}8,0{,}6)$. Оба вектора нормированы. Какова их косинусная близость?

A. $0$.

B. $0{,}6$.

C. $0{,}8$.

D. $1$.




### 3. В столбце оценок для запроса $t$ стоят значения: снимок A — $0{,}8$, снимок B — $0$. Какой снимок займёт первое место?

A. Снимок A.

B. Снимок B.

C. Оба снимка, поскольку оценки надо сложить.

D. Нельзя определить без вероятностей классов.




### 4. Какую форму имеет матрица оценок для трёх изображений и одного описания?

A. $3	imes1$.

B. $1	imes3$.

C. $3	imes3$.

D. $1	imes1$.




### 5. Есть одно изображение и три описания. Как найти описание с наибольшей оценкой?

A. Матрица имеет форму $3	imes1$; выбрать строку с максимумом.

B. Матрица имеет форму $1	imes3$; выбрать столбец с максимумом в единственной строке.

C. Матрица имеет форму $3	imes3$; выбрать диагональ.

D. Матрица имеет форму $1	imes1$; три оценки объединяются в одну.




### 6. Вы сравниваете два текста запроса. Как провести сравнение, чтобы изменение ранга можно было связать с формулировкой?

A. Оставить изображения и модель прежними, изменить только текст.

B. Одновременно заменить текст, изображения и модель.

C. Добавить новые снимки только для второго текста.

D. Изменить порядок изображений и сравнить сырые оценки разных запросов.




### 7. Для одной фотографии вычислили softmax по трём собственным описаниям. Что можно заключить из полученных долей?

A. Это вероятности истинных классов, не зависящие от списка описаний.

B. Наибольшая доля доказывает, что соответствующий предмет есть на фотографии.

C. Это относительные доли среди данных описаний; замена списка может изменить их.

D. Их можно сравнивать как вероятности между разными фотографиями без дополнительных условий.




### 8. Запрос «a photo of a bicycle» не соответствует ни одному изображению в галерее, но поиск всё равно выдал лидера. Какой вывод и действие уместны?

A. Считать лидера подходящим, потому что он первый.

B. Проверить изображения и предусмотреть отказ; для автоматического порога нужны размеченные примеры с совпадениями и без них.

C. Принять максимальную оценку за вероятность и отказаться только при значении меньше нуля.

D. Повторить запрос без изменений, пока лидер не сменится.




### 9. Как проверить, подходит ли англоязычная версия CLIP для запросов на русском или белорусском?

A. Перевести один запрос и считать совпадение поддержки языка доказанным.

B. На тех же изображениях сравнить запросы на каждом языке и вручную оценить результаты на нескольких примерах.

C. Проверить только форму матрицы: если она не изменилась, языки поддерживаются одинаково.

D. Сравнить максимальные сырые оценки между языками как вероятности.




### 10. Для трёх запросов с подходящими изображениями правильный снимок занял места 1, 4 и 2. Какова доля попаданий в top-3?

A. $1/3$.

B. $3/3$.

C. $2/3$.

D. $2/4$.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Модель переводит оба вида входа в представления, которые можно сравнить.



**Решение**


**B.** CLIP строит векторы изображения и текста, затем вычисляет оценку их совместимости.



### Вопрос 2

**Подсказка**


Для единичных векторов косинус равен скалярному произведению.



**Решение**


**C.** $(1,0)\cdot(0{,}8,0{,}6)=0{,}8$. Вторые координаты не дают вклада, поскольку у запроса там ноль.



### Вопрос 3

**Подсказка**


Для одного запроса сравните оценки кандидатов в его столбце.



**Решение**


**A.** $0{,}8>0$, поэтому снимок A будет выше снимка B в ранжированном списке. Это не доказывает, что A подходит.



### Вопрос 4

**Подсказка**


В `logits_per_image` строка относится к изображению, столбец — к тексту.



**Решение**


**A.** Каждому из трёх изображений соответствует одна оценка, поэтому матрица имеет три строки и один столбец.



### Вопрос 5

**Подсказка**


Число строк задаёт число изображений, число столбцов — число текстов.



**Решение**


**B.** Единственная строка содержит три оценки, по одной на описание. Выберите наибольшую из них.



### Вопрос 6

**Подсказка**


В опыте меняют одно условие за раз.



**Решение**


**A.** При одинаковых снимках и модели сравнение показывает, как новая формулировка изменила ранжирование.



### Вопрос 7

**Подсказка**


В знаменатель softmax входят оценки всех предложенных вариантов.



**Решение**


**C.** Softmax распределяет доли внутри выбранного списка. Эти доли сами по себе не доказывают наличие объекта и не являются безусловными вероятностями.



### Вопрос 8

**Подсказка**


Ранжирование всегда упорядочивает кандидатов, даже если все они неподходящие.



**Решение**


**B.** Первое место означает только лидерство среди кандидатов. Чтобы система могла отказываться, правило отказа следует проверить на примерах с подходящими и неподходящими изображениями.



### Вопрос 9

**Подсказка**


Нужны одинаковые кандидаты и проверка того, какие снимки действительно подходят.



**Решение**


**B.** Такой опыт позволяет сравнить результаты на выбранных примерах. Для вывода о качестве языка понадобится больше размеченных запросов и изображений.



### Вопрос 10

**Подсказка**


Попаданием считается место 1, 2 или 3. Сначала посчитайте такие случаи, затем разделите на число запросов.



**Решение**


**C.** В top-3 попали первый и третий результаты, поэтому доля равна $2/3$. Это небольшая учебная проверка, а не общая оценка качества поиска.



